# TP — Programmation orientée objet en Python
## Simulation d’un écosystème proie-prédateur

**Rendu final**

Ce notebook rassemble les éléments demandés pour le rendu du TP :

1. présentation de la conception ;
2. diagramme UML ;
3. tests unitaires exécutables ;
4. rapport de conception ;
5. démonstration de la simulation.

Le code complet de la simulation reste dans le fichier `POO_simulation_vie.py`. Les cellules de code ci-dessous n'en recopient que des extraits nécessaires pour illustrer la conception ou exécuter les tests.


## 1. Présentation générale

L'objectif du programme est de simuler un petit écosystème composé de plantes, d'herbivores et de carnivores sur une grille.

Au cours de la simulation, les animaux :

- se déplacent ;
- consomment des réserves ;
- vieillissent ;
- peuvent se reproduire ;
- les carnivores peuvent chasser les herbivores ;
- les individus dont les réserves sont épuisées peuvent mourir ;
- les plantes peuvent repousser.

Le modèle n'a pas pour objectif de représenter fidèlement un écosystème réel. Il s'agit avant tout d'utiliser les principes de la programmation orientée objet dans une simulation simple.

Dans l'implémentation finale, les principales classes sont `plante`, `animale`, `herbivore` et `carnivore`. La gestion globale de la simulation est réalisée par les fonctions `Simulation()`, `run_simulation()` et `repousse_plantes()`, tandis que `lancer_dashboard()` s'occupe de la visualisation.

PS : Toute la visualisation a été générée par IA


## 2. Architecture objet

La hiérarchie effectivement utilisée dans le programme est :

```text
                         animale
                        /       \
                       /         \
                herbivore     carnivore
                    |              |
                 lapin           renard
```

`herbivore` et `carnivore` héritent donc de `animale`.

La classe `plante` est indépendante de cette hiérarchie.

> **Différence avec le cahier des charges final :** l'énoncé proposait explicitement les classes `Animal`, `Proie`, `Predateur`, `Lapin`, `Loup`, `Environnement` et `Simulation`. Le programme réalisé utilise une architecture plus compacte : `animale` remplace la classe générale `Animal`, `herbivore` et `carnivore` regroupent directement les deux types d'animaux, et la gestion de l'environnement/simulation est assurée par des fonctions plutôt que par une classe `Environnement`.


## 3. Diagramme UML

# Diagramme UML de la simulation

<div class="mermaid">

classDiagram
    direction TB

    class plante {
        +NOURRITURE: int = 12
        +pos: tuple
        +nourriture: float
        +vivant: bool
        +__init__(pos)
        +mort()
    }

    class animale {
        <<classe de base>>
        +COUT_DEP: float
        +AGE_MATURE: int
        +AGE_VIEU: int
        +ESPERANCE: int
        +SENES_K: float
        +SENES_P: float
        +COUT_JEUNE: float
        +REPRO_AGE: int
        +REPRO_SEUIL: float
        +REPRO_GARDE: float
        +REPRO_PART: float
        +REPRO_PAUSE: int
        +SUCCES_CHASSE: float
        +PART_CORPS: float
        +pos: tuple
        +espece: str
        +vitesse: int
        +vision: int
        +rmax: float
        +nourriture: float
        +trepro: int
        +age: int
        +vivant: bool
        +__init__(pos, espece, vitesse, vision, rmax, nourriture)
        +facteur_age() float
        +deplacement(cibles)
        +manger(cible)
        +mort()
        +repro() animale
    }

    class herbivore {
        <<sous-classe>>
        +NOURRITURE_INIT: float = 0.4
        +__init__(pos, espece, vitesse, vision, rmax, nourriture)
        +action(plante_list) animale
    }

    class carnivore {
        <<sous-classe>>
        +NOURRITURE_INIT: float = 0.8
        +__init__(pos, espece, vitesse, vision, rmax, nourriture)
        +action(herbivore_list) animale
    }

    class FonctionsSimulation {
        <<fonctions globales>>
        +initialisation(nplante, nherbivore, ncarnivore)
        +Simulation(plante_list, herbivore_list, carnivore_list)
        +run_simulation(...)
        +repousse_plantes(plante_list, plante_max, taux_repousse)
        +dep_age(...)
        +proportion_reserve_proie(r_proie, r)
    }

    class Dashboard {
        <<interface graphique>>
        +lancer_dashboard(history)
        +maj_affichage(jour)
        +toggle_play(event)
        +step_prev(event)
        +step_next(event)
        +on_slide(val)
    }

    animale <|-- herbivore : herite
    animale <|-- carnivore : herite

    FonctionsSimulation ..> plante : cree / filtre
    FonctionsSimulation ..> herbivore : cree / simule
    FonctionsSimulation ..> carnivore : cree / simule
    FonctionsSimulation ..> animale : utilise les comportements

    herbivore ..> plante : chasse / mange
    carnivore ..> herbivore : chasse / mange
    animale ..> plante : cible possible
    animale ..> animale : se reproduit

    Dashboard ..> FonctionsSimulation : affiche history
    Dashboard ..> plante : affiche positions
    Dashboard ..> herbivore : affiche positions, ages, reserves
    Dashboard ..> carnivore : affiche positions, ages, reserves

</div>

<script type="module">
import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
mermaid.initialize({ startOnLoad: true });
</script>


### Relations importantes

- **Héritage :** `herbivore` et `carnivore` héritent de `animale`.
- **Composition / agrégation des objets du monde :** les listes de simulation contiennent les objets `plante`, `herbivore` et `carnivore`.
- **Interaction :** les carnivores reçoivent une liste d'herbivores comme cibles et peuvent tuer un herbivore.
- **Spécialisation :** les classes filles redéfinissent plusieurs paramètres de la classe `animale` et possèdent chacune leur propre méthode `action()`.


## 4. Rapport de conception

### 4.1 Héritage

L'héritage est utilisé pour éviter de dupliquer les caractéristiques communes aux différents animaux.

La classe `animale` contient notamment :

- la position ;
- la vitesse ;
- la vision ;
- les réserves de nourriture ;
- l'âge ;
- l'état vivant/mort ;
- les comportements généraux de déplacement, de nourriture, de mort et de reproduction.

Les classes `herbivore` et `carnivore` héritent de cette classe :

```python
class herbivore(animale):
    ...

class carnivore(animale):
    ...
```

Elles peuvent ainsi réutiliser le comportement commun tout en spécialisant certains paramètres et certaines actions.

Par exemple, les deux classes ont des âges de maturité et des paramètres de reproduction différents. Le carnivore possède également une probabilité de réussite de chasse et une valeur associée au corps de la proie.

Ce choix permet donc de centraliser le comportement commun et de ne modifier que ce qui est spécifique à chaque type d'animal.


### 4.2 Polymorphisme

Le polymorphisme apparaît notamment avec la méthode `action()`.

Les deux classes filles possèdent une méthode portant le même nom :

```python
herbivore.action(plante_list)
carnivore.action(herbivore_list)
```

mais leur comportement est différent.

L'herbivore utilise les plantes comme cibles alors que le carnivore utilise les herbivores.

Cela permet de conserver une interface commune tout en laissant chaque classe définir son comportement spécifique.

Le polymorphisme permet ainsi d'éviter de concentrer toute la logique dans une succession de tests du type :

```python
if isinstance(animal, herbivore):
    ...
elif isinstance(animal, carnivore):
    ...
```

Le comportement est directement défini dans la classe de l'objet.


On a également un paramètre modifiable de `vitesse`, `vision`, `rmax` (capacité de reserve maximale) ... qui pourrait servir pour créer des populations non homogène (du polymorphisme). On pourrait ensuite penser à la reproduction, en faisant en sorte que les enfants aient des genes qui valent la moyenne des parents +/- une mutation aléaoire.


### 4.3 Encapsulation

L'encapsulation consiste à faire gérer son propre état par l'objet plutôt que de modifier arbitrairement ses données depuis l'extérieur.

Dans le programme, les réserves de nourriture sont modifiées par les méthodes de l'animal lors des différentes actions.

Par exemple, pendant un déplacement :

```python
cout = self.facteur_age() * self.rmax * self.COUT_DEP
self.nourriture = max(0, self.nourriture - cout)
```

La valeur ne peut donc pas devenir négative à la suite de cette opération.

Lorsqu'un animal gagne de la nourriture, la valeur est également limitée par `rmax` :

```python
self.nourriture = min(self.rmax, self.nourriture + gain)
```

L'encapsulation est toutefois **partielle** dans cette version : les attributs restent directement accessibles en Python. Une version plus poussée pourrait utiliser des propriétés ou des attributs privés avec des méthodes `perdre_energie()` et `gagner_energie()`.


### 4.4 Composition

La simulation manipule des collections d'objets représentant les différents éléments de l'écosystème :

```text
Simulation
├── plante_list
├── herbivore_list
└── carnivore_list
```

Ces listes contiennent les instances des différentes classes.

La fonction `Simulation()` parcourt ces collections, fait agir les animaux, puis actualise les populations en supprimant les individus morts et en ajoutant les nouveau-nés.

L'énoncé demandait idéalement une classe `Environnement` contenant une collection d'animaux. Dans le programme réalisé, cette responsabilité est plutôt répartie entre les fonctions de simulation et les différentes listes.


### 4.5 Répartition des responsabilités

| Élément | Responsabilité |
|---|---|
| `plante` | Représenter une plante, sa réserve de nourriture et son état vivant/mort |
| `animale` | Regrouper les caractéristiques et comportements communs aux animaux |
| `herbivore` | Spécialiser le comportement des animaux qui recherchent les plantes |
| `carnivore` | Spécialiser le comportement des animaux qui recherchent et chassent les herbivores |
| `initialisation()` | Créer les populations initiales |
| `Simulation()` | Faire évoluer les populations pendant un tour |
| `dep_age()` | Calculer l'effet de l'âge sur le coût du déplacement |
| `proportion_reserve_proie()` | Calculer la part de ressources récupérée à partir d'une proie |
| `repousse_plantes()` | Ajouter de nouvelles plantes pour simuler leur repousse |
| `run_simulation()` | Exécuter la simulation pendant plusieurs jours et enregistrer son historique |
| `lancer_dashboard()` | Afficher l'évolution de la simulation et permettre sa lecture interactive |


## 5. Fonctionnement d'un tour

Un tour de simulation est réalisé par `Simulation()`.

L'ordre général est :

1. chaque herbivore vivant réalise son `action()` ;
2. chaque carnivore vivant réalise son `action()` ;
3. les individus morts sont retirés des listes ;
4. les nouveaux individus issus de la reproduction sont ajoutés.

Puis, dans `run_simulation()`, les plantes repoussent et un nouvel état de la simulation est enregistré.

Cette organisation permet de séparer le fonctionnement individuel d'un animal (`action()`, `deplacement()`, `manger()`, `repro()`) de la gestion globale des populations.


## 6. Paramètres de la simulation utilisée pour la démonstration

La simulation lancée dans le fichier Python utilise :

| Paramètre | Valeur |
|---|---:|
| Taille de la carte | `100 × 100` |
| Plantes initiales | `600` |
| Herbivores initiaux | `100` |
| Carnivores initiaux | `25` |
| Nombre de jours | `400` |
| Nombre maximal de plantes | `800` |
| Taux de repousse | `0,22` |

Ces valeurs correspondent à l'appel placé à la fin du fichier Python.


# 7. Tests unitaires

Les tests suivants sont exécutables directement depuis ce notebook.

Ils importent les classes et fonctions du fichier `POO_simulation_vie.py`. Comme le fichier contient son lancement graphique dans :

```python
if __name__ == "__main__":
```

l'import ne lance pas automatiquement le dashboard.

Les tests ci-dessous vérifient les comportements correspondant aux mécanismes réellement présents dans le programme.


In [ ]:
import sys
from pathlib import Path
import unittest

sys.path.insert(0, str(Path.cwd()))

import POO_simulation_vie as sim


### Test 1 — Création d'un animal

On vérifie qu'un herbivore possède bien un âge initial, une réserve de nourriture positive et qu'il est vivant lors de sa création.


In [ ]:
class TestAnimale(unittest.TestCase):
    def test_creation_herbivore(self):
        h = sim.herbivore((10, 20), "lapin", 3, 9, 20)
        self.assertEqual(h.age, 1)
        self.assertEqual(h.nourriture, 20 * h.NOURRITURE_INIT)
        self.assertTrue(h.vivant)


### Test 2 — Vieillissement

L'action d'un herbivore augmente son âge de 1 et augmente également son compteur depuis la dernière reproduction. Le déplacement consomme une partie de ses réserves.


In [ ]:
class TestVieillissement(unittest.TestCase):
    def test_action_fait_vieillir(self):
        h = sim.herbivore((10, 20), "lapin", 0, 9, 20)
        age_initial = h.age
        nourriture_initiale = h.nourriture

        h.action([])

        self.assertEqual(h.age, age_initial + 1)
        self.assertLess(h.nourriture, nourriture_initiale)


### Test 3 — Mort

La méthode `mort()` doit faire passer l'attribut `vivant` à `False`.


In [ ]:
class TestMort(unittest.TestCase):
    def test_mort(self):
        h = sim.herbivore((10, 20), "lapin", 0, 9, 20)
        h.mort()
        self.assertFalse(h.vivant)


### Test 4 — Déplacement

On vérifie que le déplacement modifie effectivement la position et respecte le comportement de déplacement vers une cible.


In [ ]:
class TestDeplacement(unittest.TestCase):
    def test_deplacement_vers_plante(self):
        h = sim.herbivore((10, 10), "lapin", 1, 9, 20)
        p = sim.plante((11, 10))

        position_initiale = h.pos
        h.deplacement([p])

        self.assertNotEqual(h.pos, position_initiale)


### Test 5 — Chasse

On place un carnivore et un herbivore sur la même position. La probabilité de réussite de chasse est temporairement fixée à `1` pour rendre le test déterministe.


In [ ]:
class TestChasse(unittest.TestCase):
    def test_chasse(self):
        c = sim.carnivore((10, 10), "renard", 0, 12, 30, nourriture=5)
        h = sim.herbivore((10, 10), "lapin", 0, 9, 20, nourriture=5)

        ancien_succes = c.SUCCES_CHASSE
        c.SUCCES_CHASSE = 1.0

        try:
            c.manger(h)
            self.assertFalse(h.vivant)
            self.assertGreater(c.nourriture, 5)
        finally:
            c.SUCCES_CHASSE = ancien_succes


### Test 6 — Collections de la simulation

On vérifie que l'initialisation produit bien le nombre demandé de plantes, d'herbivores et de carnivores.


In [ ]:
class TestInitialisation(unittest.TestCase):
    def test_nombre_individus(self):
        plantes, herbivores, carnivores = sim.initialisation(20, 5, 2)

        self.assertEqual(len(plantes), 20)
        self.assertEqual(len(herbivores), 5)
        self.assertEqual(len(carnivores), 2)


### Test 7 — Suppression des morts

La fonction `Simulation()` doit retirer des listes les individus morts.


In [ ]:
class TestSuppression(unittest.TestCase):
    def test_suppression_morts(self):
        plantes = []
        herbivores = [
            sim.herbivore((10, 10), "lapin", 0, 9, 20)
        ]
        carnivores = []

        herbivores[0].mort()

        sim.Simulation(plantes, herbivores, carnivores)

        self.assertEqual(len(herbivores), 0)


### Test 8 — Reproduction

On vérifie qu'un animal disposant de suffisamment de réserves et ayant atteint l'âge nécessaire peut produire un nouvel individu.

Le test fixe le compteur de reproduction pour satisfaire la condition du modèle.


In [ ]:
class TestReproduction(unittest.TestCase):
    def test_reproduction(self):
        h = sim.herbivore((10, 10), "lapin", 0, 9, 20, nourriture=20)
        h.age = h.AGE_MATURE
        h.trepro = h.REPRO_PAUSE + 1

        enfant = h.repro()

        self.assertIsNotNone(enfant)
        self.assertEqual(enfant.espece, h.espece)
        self.assertEqual(enfant.age, 1)


### Exécution de tous les tests

La cellule suivante lance les tests définis dans ce notebook.


In [ ]:
suite = unittest.TestLoader().loadTestsFromModule(sys.modules[__name__])
result = unittest.TextTestRunner(verbosity=2).run(suite)

assert result.wasSuccessful()


## 8. Test rapide de la simulation complète

Avant la démonstration graphique, on peut exécuter une petite simulation sans afficher le dashboard afin de vérifier que la fonction retourne bien un historique.


In [ ]:
history_test = sim.run_simulation(
    nplante=100,
    nherbivore=20,
    ncarnivore=5,
    NB_JOURS=30,
    plante_max=120,
    taux_repousse=0.22
)

print("Nombre d'états enregistrés :", len(history_test))
print("État initial :", {
    "plantes": len(history_test[0]["plantes_pos"]),
    "herbivores": len(history_test[0]["herbivores"]),
    "carnivores": len(history_test[0]["carnivores"])
})
print("Dernier état :", {
    "jour": history_test[-1]["jour"],
    "plantes": len(history_test[-1]["plantes_pos"]),
    "herbivores": len(history_test[-1]["herbivores"]),
    "carnivores": len(history_test[-1]["carnivores"])
})


# 9. Démonstration de la simulation

Le fichier Python contient déjà le lancement de la simulation avec les paramètres du projet et le dashboard interactif.

La cellule suivante reprend le même principe directement dans le notebook. Elle permet d'obtenir :

- la carte de l'écosystème ;
- l'évolution des populations ;
- la répartition des âges ;
- la répartition des réserves de nourriture ;
- un curseur permettant de parcourir les jours ;
- des boutons de lecture, précédent et suivant.


In [ ]:
history = sim.run_simulation(
    nplante=600,
    nherbivore=100,
    ncarnivore=25,
    NB_JOURS=400,
    plante_max=800,
    taux_repousse=0.22
)

sim.lancer_dashboard(history)


# 10. Réponses aux questions finales

## Q1 — Quelle différence entre une classe et un objet ?

Une classe est un modèle qui définit les caractéristiques et les comportements d'un type d'objet. Un objet est une instance concrète créée à partir de cette classe.

Par exemple, `herbivore` est une classe tandis qu'un individu créé avec `herbivore(...)` est un objet.

## Q2 — Pourquoi Lapin peut-il hériter de Proie ?

Un lapin est un type de proie. Il peut donc récupérer les caractéristiques et comportements communs aux proies tout en ajoutant ou modifiant certains comportements spécifiques.

Dans le programme réalisé, cette distinction est simplifiée : la classe `herbivore` représente directement cette catégorie d'animaux.

## Q3 — Pourquoi Loup peut-il hériter de Predateur ?

Un loup est un type de prédateur. Il peut donc hériter des caractéristiques communes aux prédateurs et spécialiser son comportement de chasse.

Dans le programme réalisé, cette catégorie est représentée directement par `carnivore`.

## Q4 — Exemple d'héritage

```python
class herbivore(animale):
    ...

class carnivore(animale):
    ...
```

Les deux classes héritent de `animale`.

## Q5 — Exemple de composition

Les populations de la simulation sont représentées par des listes contenant des objets :

```text
plante_list
herbivore_list
carnivore_list
```

La simulation manipule donc des ensembles d'objets constituant l'écosystème.

## Q6 — Exemple de polymorphisme

`herbivore` et `carnivore` possèdent tous les deux une méthode `action()`, mais celle-ci n'effectue pas la même opération selon la classe.

L'herbivore agit vis-à-vis des plantes tandis que le carnivore agit vis-à-vis des herbivores.

## Q7 — Pourquoi Animal peut-il être abstrait ?

Une classe générale `Animal` peut être abstraite lorsqu'elle sert principalement à définir une structure commune sans représenter nécessairement un animal concret.

Certaines méthodes peuvent être communes et d'autres imposées aux classes filles.

Dans le code réalisé, `animale` n'est toutefois pas déclarée comme classe abstraite avec `ABC` et `@abstractmethod`.

## Q8 — Pourquoi protéger la modification de l'énergie ?

Faire modifier l'état interne par l'objet permet de contrôler les valeurs et d'éviter des états incohérents. Une méthode comme `perdre_energie()` pourrait par exemple empêcher une réserve négative.

Dans le code réalisé, cette protection est partiellement présente avec `max(0, ...)` et `min(self.rmax, ...)`.

## Q9 — Que se passerait-il sans Environnement ?

La simulation pourrait toujours fonctionner, mais les responsabilités seraient davantage concentrées dans le programme principal. Cela rendrait la logique plus difficile à organiser, tester et modifier.

Le programme réalisé évite partiellement ce problème en séparant les fonctions d'initialisation, de simulation, de repousse et d'affichage.

## Q10 — Quel principe est le plus difficile à comprendre ?

Le polymorphisme peut être difficile à comprendre car il ne consiste pas uniquement à faire hériter une classe d'une autre. Il faut comprendre que plusieurs classes peuvent fournir une même interface tout en ayant des comportements différents.

Dans ce projet, la différence entre les méthodes `action()` des herbivores et des carnivores illustre concrètement cette idée.


# 11. Bilan par rapport au cahier des charges

Le programme met en œuvre les principaux mécanismes demandés : classes, objets, héritage, spécialisation, interactions, gestion des populations, reproduction, chasse et visualisation.

Certains éléments du cahier des charges final sont cependant implémentés sous une forme différente :

- `Animal` est représentée par `animale` ;
- `Proie` et `Predateur` ne sont pas des classes intermédiaires ;
- `Lapin` et `Loup` sont représentés par `herbivore` et `carnivore` ;
- `Environnement` n'est pas une classe distincte ;
- `Simulation` est une fonction et non une classe ;
- `Animal` n'est pas une classe abstraite Python ;
- l'encapsulation est présente mais n'utilise pas de propriétés privées ;
- les tests sont ici fournis sous forme de `unittest` dans le notebook.

Cette architecture reste cohérente avec l'objectif général du TP : utiliser la programmation orientée objet pour représenter les différents éléments d'un écosystème et leurs interactions.


# 12. Livrables

- **Code Python complet :** `POO_simulation_vie.py`
- **Diagramme UML :** section 3 de ce notebook
- **Tests unitaires :** section 7, exécutables directement
- **Rapport de conception :** sections 4 et 11
- **Démonstration de la simulation :** section 9
